In [0]:
pip install paramiko

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import paramiko
import os
import io
import boto3


# ---- Config ----
AWS_REGION   = "us-east-1"
SECRET_SCOPE = "supporthub-dev"
KNOWN_HOSTS  = "/Volumes/workspace/default/aws_sftp/known_hosts"

# ---- AWS clients ----
session = boto3.Session(
    aws_access_key_id=dbutils.secrets.get(SECRET_SCOPE, "aws_access_key_id"),
    aws_secret_access_key=dbutils.secrets.get(SECRET_SCOPE, "aws_secret_access_key"),
    region_name=AWS_REGION,
)
ec2 = session.client("ec2")
ssm = session.client("ssm")

INSTANCE_ID   = ssm.get_parameter(Name="/supporthub/dev/sftp/instance_id")["Parameter"]["Value"]
PRIVATE_KEY   = dbutils.secrets.get("aws-sftp", "private_key")
SFTP_USERNAME = dbutils.secrets.get("aws-sftp", "sftp_username")


def get_public_ip(instance_id: str) -> str:
    """Start the instance if needed and return its current public IPv4."""
    instance = ec2.describe_instances(InstanceIds=[instance_id])["Reservations"][0]["Instances"][0]
    state = instance["State"]["Name"]

    if state == "stopped":
        ec2.start_instances(InstanceIds=[instance_id])
    if state != "running":
        ec2.get_waiter("instance_running").wait(InstanceIds=[instance_id])
        ec2.get_waiter("instance_status_ok").wait(InstanceIds=[instance_id])
        instance = ec2.describe_instances(InstanceIds=[instance_id])["Reservations"][0]["Instances"][0]

    ip = instance.get("PublicIpAddress")
    if not ip:
        raise RuntimeError(f"{instance_id} has no public IPv4")
    return ip


def ensure_known_host(ip: str, port: int = 22, allow_first_use: bool = False) -> None:
    """Make sure `ip` is in known_hosts. A new IP is only added if the server
    presents a host key that is already trusted under another name/IP."""
    if paramiko.HostKeys(KNOWN_HOSTS).lookup(ip):
        return

    # Fetch the key the server at this IP presents
    t = paramiko.Transport((ip, port))
    try:
        t.start_client(timeout=30)
        server_key = t.get_remote_server_key()
    finally:
        t.close()
    key_type, key_b64 = server_key.get_name(), server_key.get_base64()

    # Read the existing file and collect all trusted keys
    with open(KNOWN_HOSTS) as f:
        content = f.read()

    trusted = set()
    for line in content.splitlines():
        parts = line.split()
        if len(parts) >= 3 and not line.startswith(("#", "@")):
            trusted.add((parts[1], parts[2]))

    if (key_type, key_b64) not in trusted and not (allow_first_use and not trusted):
        raise RuntimeError(
            f"Host key from {ip} does not match any trusted key in {KNOWN_HOSTS}. "
            "Verify the server's fingerprint before trusting it."
        )

    # Volumes don't support append mode - rewrite the whole file
    if content and not content.endswith("\n"):
        content += "\n"
    with open(KNOWN_HOSTS, "w") as f:
        f.write(content + f"{ip} {key_type} {key_b64}\n")
    print(f"Added {ip} to known_hosts")


# ---- Load private key ----
private_key = io.StringIO(PRIVATE_KEY)
try:
    pkey = paramiko.Ed25519Key.from_private_key(private_key)
except paramiko.SSHException:
    private_key.seek(0)
    pkey = paramiko.RSAKey.from_private_key(private_key)

# ---- Connect ----
hostname = get_public_ip(INSTANCE_ID)
print("Connecting to", hostname)
ensure_known_host(hostname)

ssh = paramiko.SSHClient()
ssh.load_host_keys(KNOWN_HOSTS)
ssh.set_missing_host_key_policy(paramiko.RejectPolicy())
ssh.connect(hostname=hostname, port=22, username=SFTP_USERNAME, pkey=pkey, timeout=30)
sftp = ssh.open_sftp()
print("Successfully connected to AWS SFTP!")


Connecting to 34.236.144.253
Successfully connected to AWS SFTP!


In [0]:
import os
import posixpath
import stat
from datetime import datetime, timezone

# ---- Config ----
source_folders = {
    "forecasts": "/data/wfm/forecasts",
    "schedules": "/data/wfm/schedules",
    "remediation": "/data/wfm/remediation",
}
base_path = "/Volumes/workspace/default/aws_sftp"
LOG_TABLE = "workspace.default.sftp_ingestion_log"
MIN_FILE_AGE_SECONDS = 120   # skip files modified in the last 2 minutes (may still be uploading)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
        remote_path  STRING,
        local_path   STRING,
        size         BIGINT,
        remote_mtime BIGINT,
        ingested_at  TIMESTAMP
    )
""")


def list_remote_files(sftp, remote_path):
    """Recursively yield (full_path, attributes) for every file under remote_path."""
    for item in sftp.listdir_attr(remote_path):
        full_path = posixpath.join(remote_path, item.filename)
        if stat.S_ISDIR(item.st_mode):
            yield from list_remote_files(sftp, full_path)
        elif stat.S_ISREG(item.st_mode):
            yield full_path, item


try:
    # Files already ingested: remote_path -> (size, mtime)
    ingested = {
        r.remote_path: (r.size, r.remote_mtime)
        for r in spark.table(LOG_TABLE).collect()
    }

    new_rows = []
    downloaded = skipped = in_progress = 0
    now = datetime.now(timezone.utc).timestamp()

    for folder, remote_root in source_folders.items():
        local_root = f"{base_path}/{folder}"
        print(f"\nChecking: {folder}")

        for remote_file, attr in list_remote_files(sftp, remote_root):
            signature = (attr.st_size, attr.st_mtime)

            # Already ingested and unchanged -> skip
            if ingested.get(remote_file) == signature:
                skipped += 1
                continue

            # Recently modified -> may still be uploading, pick it up next run
            if now - attr.st_mtime < MIN_FILE_AGE_SECONDS:
                print(f"Still being written, skipping for now: {remote_file}")
                in_progress += 1
                continue

            rel_path = posixpath.relpath(remote_file, remote_root)
            local_file = os.path.join(local_root, rel_path)

            # First run: file already downloaded by the earlier full load -> just record it
            if (
                remote_file not in ingested
                and os.path.exists(local_file)
                and os.path.getsize(local_file) == attr.st_size
            ):
                print(f"Already present, recording: {remote_file}")
            else:
                os.makedirs(os.path.dirname(local_file), exist_ok=True)
                print(f"Downloading: {remote_file}")
                sftp.get(remote_file, local_file)
                downloaded += 1

            new_rows.append((remote_file, local_file, attr.st_size, attr.st_mtime,
                             datetime.now(timezone.utc)))

    # Update the log (one row per file, updated if the file changed)
    if new_rows:
        spark.createDataFrame(
            new_rows,
            "remote_path STRING, local_path STRING, size BIGINT, remote_mtime BIGINT, ingested_at TIMESTAMP",
        ).createOrReplaceTempView("new_files")

        spark.sql(f"""
            MERGE INTO {LOG_TABLE} t
            USING new_files s
            ON t.remote_path = s.remote_path
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

    print(
        f"\nDownloaded: {downloaded} | Unchanged: {skipped} | "
        f"In progress: {in_progress} | Logged: {len(new_rows)}"
    )

finally:
    # Always close the connection, even if something failed
    sftp.close()
    ssh.close()



Checking: forecasts

Checking: schedules

Checking: remediation

Downloaded: 0 | Unchanged: 66 | In progress: 0 | Logged: 0
